<a href="https://colab.research.google.com/github/Wanuzia/simulate/blob/main/enade_20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pypdf


In [ ]:
import re
import json
from pypdf import PdfReader

In [ ]:
leitor = PdfReader('/2021_PV_tecnologia_analise_desenvolvimento_sistemas.pdf')

In [ ]:
texto_completo = ""
for pagina in leitor.pages:
    texto_completo += pagina.extract_text() + "\n"

In [ ]:
padrao_questao = r'(QUESTÃO\s+(?:DISCURSIVA\s+)?\d+)'

In [ ]:
partes = re.split(padrao_questao, texto_completo)

In [ ]:
instrucoes_iniciais = partes[0].strip()

questoes_estruturadas = []

In [ ]:
for i in range(1, len(partes), 2):
    titulo_questao = partes[i].strip()
    corpo_questao = partes[i+1].strip() if (i+1) < len(partes) else ""

    # Remove as linhas de "RASCUNHO" e números de 1 a 15 para limpar o JSON
    corpo_questao = re.sub(r'RASCUNHO\s+1\s+2\s+3.*Área livre', '[Espaço de Rascunho Removido]', corpo_questao, flags=re.DOTALL)

    # Adiciona a questão estruturada à nossa lista
    questoes_estruturadas.append({
        "identificador": titulo_questao,
        "texto_da_questao": corpo_questao.strip()
    })


In [ ]:
json_final = {
    "prova": "Tecnologia em Análise e Desenvolvimento de Sistemas",
    "total_paginas": len(leitor.pages),
    "instrucoes_gerais": instrucoes_iniciais,
    "questoes": questoes_estruturadas
}

In [ ]:
json_resultado = json.dumps(json_final, indent=4, ensure_ascii=False)

In [ ]:
with open("/content/prova_estruturada.json", "w", encoding="utf-8") as f:
    f.write(json_resultado)

print("JSON estruturado com sucesso! Verifique a pasta lateral do Colab para baixar o arquivo 'prova_estruturada.json'.")

JSON estruturado com sucesso! Verifique a pasta lateral do Colab para baixar o arquivo 'prova_estruturada.json'.


In [ ]:
pdf_path = '/2021_ADS_GABARITO.pdf'
import re
import json
from pypdf import PdfReader

def extrair_gabarito_corrigido(pdf_path):
    leitor_gabarito = PdfReader(pdf_path)
    texto_gabarito = ""

    for pagina in leitor_gabarito.pages:
        texto_gabarito += pagina.extract_text() + "\n"

    # Nova Regex: Procura pela palavra QUESTÃO, captura o número da questão,
    # pula os espaços e captura a letra da resposta (A a E)
    padrao_gabarito = r'QUESTÃO\s+(\d+)\s+([A-E])'

    gabarito_mapeado = {}

    for linha in texto_gabarito.split('\n'):
        match = re.search(padrao_gabarito, linha)
        if match:
            num_questao = int(match.group(1))
            resposta = match.group(2)

            # Aqui está o segredo: formatamos com :02d para que "1" vire "01", casando com seu JSON
            chave_questao = f"QUESTÃO {num_questao:02d}"
            gabarito_mapeado[chave_questao] = resposta

    return gabarito_mapeado

# --- Execução do Merge ---

# 1. Extrai o gabarito com a regex corrigida
caminho_gabarito_pdf = pdf_path
gabarito_final = extrair_gabarito_corrigido(caminho_gabarito_pdf)

print(f"Gabarito mapeado com sucesso! Mapeadas {len(gabarito_final)} questões objetivas.")
# Para debug: print(gabarito_final) deve mostrar {"QUESTÃO 01": "E", "QUESTÃO 02": "C", ...}

# 2. Carrega o seu JSON de questões atual
with open("/content/prova_estruturada.json", "r", encoding="utf-8") as f:
    dados_prova = json.load(f)

# 3. Varre a lista de questões para injetar o gabarito correto
for questao in dados_prova["questoes"]:
    identificador = questao["identificador"]

    if "DISCURSIVA" in identificador:
        questao["gabarito"] = "Discursiva"
    else:
        # Faz o match perfeito agora que as chaves estão normalizadas (ex: "QUESTÃO 01")
        questao["gabarito"] = gabarito_final.get(identificador, "Não encontrado no PDF")

# 4. Salva o arquivo final atualizado
with open("/content/prova_completa_com_gabarito.json", "w", encoding="utf-8") as f:
    json.dump(dados_prova, f, indent=4, ensure_ascii=False)

print("Arquivo 'prova_completa_com_gabarito.json' gerado com sucesso com as relações corrigidas!")

Gabarito mapeado com sucesso! Mapeadas 35 questões objetivas.
Arquivo 'prova_completa_com_gabarito.json' gerado com sucesso com as relações corrigidas!


In [48]:
# Instala a biblioteca oficial do Gemini
!pip install google-genai -q

import json
import time
from google.genai import types
from google.colab import userdata

# Inicializa o cliente do Gemini usando a chave dos Segredos do Colab
try:
    api_key_segura = userdata.get('GEMINI_API_KEY')
    from google import genai
    client = genai.Client(api_key=api_key_segura)
    print("🚀 Conectado ao Gemini com sucesso!")
except Exception as e:
    print("❌ Erro: Verifique se a chave 'GEMINI_API_KEY' está ativa nos Segredos do Colab.")

def gerar_comentario_gemini(enunciado, alternativas, gabarito):
    # Formata as alternativas para o prompt
    texto_alternativas = "\n".join([f"{letra}: {texto}" for letra, texto in alternativas.items()])

    prompt_sistema = (
        "Você é um professor universitário sênior do curso de Análise e Desenvolvimento de Sistemas (ADS). "
        "Sua tarefa é explicar de forma extremamente didática e direta questões do ENADE.\n"
        "Regras:\n"
        "1. Escreva sua resposta em formato Markdown limpo.\n"
        "2. Explique brevemente por que a alternativa correta está certa.\n"
        "3. Explique de forma muito sucinta por que as outras alternativas estão incorretas.\n"
        "4. Seja direto, comece logo na análise."
    )

    prompt_usuario = (
        f"ENUNCIADO DA QUESTÃO:\n{enunciado}\n\n"
        f"ALTERNATIVAS:\n{texto_alternativas}\n\n"
        f"GABARITO OFICIAL: Alternativa {gabarito}\n\n"
        "Gere a explicação pedagógica:"
    )

    try:
        # Utilizando o modelo estável gemini-2.5-flash
        resposta = client.models.generate_content(
            model='gemini-1.5-flash',
            contents=prompt_usuario,
            config=types.GenerateContentConfig(
                system_instruction=prompt_sistema,
                temperature=0.3
            )
        )
        return resposta.text.strip()
    except Exception as e:
        return f"Erro ao gerar o comentário com Gemini: {str(e)}"

# --- Processamento do Arquivo ---

arquivo_entrada = "prova_completa_com_gabarito.json"
arquivo_saida = "prova_final_pronta.json"

with open(arquivo_entrada, "r", encoding="utf-8") as f:
    dados_prova = json.load(f)

print(f"Iniciando o processo para {len(dados_prova['questoes'])} questões...")

for index, questao in enumerate(dados_prova["questoes"]):
    identificador = questao["identificador"]

    if "DISCURSIVA" in identificador or questao["gabarito"] == "Discursiva":
        questao["comentario_ia"] = "Esta é uma questão discursiva. Consulte o padrão de resposta oficial do INEP."
        continue

    print(f"[{index+1}/{len(dados_prova['questoes'])}] Processando: {identificador}...")

    comentario = gerar_comentario_gemini(
        enunciado=questao["texto_da_questao"],
        alternativas=questao.get("alternativas", {}),
        gabarito=questao["gabarito"]
    )

    questao["comentario_ia"] = comentario

    # CRUCIAL: Delay de 4 segundos para não estourar o limite de 15 requisições/minuto da API gratuita
    time.sleep(4)

# Salva o arquivo final enriquecido
with open(arquivo_saida, "w", encoding="utf-8") as f:
    json.dump(dados_prova, f, indent=4, ensure_ascii=False)

print("\n🎉 Arquivo 'prova_final_pronta.json' gerado com sucesso!")

🚀 Conectado ao Gemini com sucesso!
Iniciando o processo para 40 questões...
[3/40] Processando: QUESTÃO 01...
[4/40] Processando: QUESTÃO 02...
[5/40] Processando: QUESTÃO 03...
[6/40] Processando: QUESTÃO 04...
[7/40] Processando: QUESTÃO 05...
[8/40] Processando: QUESTÃO 06...
[9/40] Processando: QUESTÃO 07...
[10/40] Processando: QUESTÃO 08...
[14/40] Processando: QUESTÃO 09...
[15/40] Processando: QUESTÃO 10...
[16/40] Processando: QUESTÃO 11...
[17/40] Processando: QUESTÃO 12...
[18/40] Processando: QUESTÃO 13...
[19/40] Processando: QUESTÃO 14...
[20/40] Processando: QUESTÃO 15...
[21/40] Processando: QUESTÃO 16...
[22/40] Processando: QUESTÃO 17...
[23/40] Processando: QUESTÃO 18...
[24/40] Processando: QUESTÃO 19...
[25/40] Processando: QUESTÃO 20...
[26/40] Processando: QUESTÃO 21...
[27/40] Processando: QUESTÃO 22...
[28/40] Processando: QUESTÃO 23...
[29/40] Processando: QUESTÃO 24...
[30/40] Processando: QUESTÃO 25...
[31/40] Processando: QUESTÃO 26...
[32/40] Processando: 

In [49]:
printArchie = "prova_final_pronta.json"
with open(printArchie, "r", encoding="utf-8") as f:
    dados_prova = json.load(f)
print(json.dumps(dados_prova, indent=4, ensure_ascii=False))

{
    "prova": "Tecnologia em Análise e Desenvolvimento de Sistemas",
    "total_paginas": 48,
    "instrucoes_gerais": "3. Verifi  que se a prova está completa e se o seu nome está correto no CArTÃo-reSPoSTA. Caso contrário, \navise imediatamente ao Chefe de Sala.\n4. Assine o CArTÃo-reSPoSTA no local apropriado, com caneta esferográfi  ca de ti  nta preta, fabricada \nem material transparente.\n5. As respostas da prova objeti  va, da prova discursiva e do questi  onário de percepção da prova deverão ser \ntranscritas, com caneta esferográfica de tinta preta, fabricada em material transparente, no \nCArTÃo-reSPoSTA que deverá ser entregue ao Chefe de Sala ao término da prova.\n6. Responda cada questão discursiva em, no máximo, 15 linhas. Qualquer texto que ultrapasse o espaço \ndesti  nado à resposta será desconsiderado.\n7. Você terá quatro horas para responder às questões de múlti  pla escolha, às questões discursivas e ao \nquesti  onário de percepção da prova.\n8. Ao terminar a pr